# Guide05: Cross-Validation, Regularization, and Grid Search

Fifth of six notebooks on one running example: predicting Ames house prices. This one finishes **Stage 9** of the [13-stage workflow](../Guide00_Supervised-ML_Linear_Regression_end-to-end_workflow.md): use cross-validation properly, control the flexibility `Guide04` showed could run away, and search for a configuration worth freezing.

**Prerequisite:** `Guide04_Supervised-ML_Linear-Regression_Polynomial-Features-and-Overfitting.ipynb`.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 15)
plt.rcParams["figure.figsize"] = (7, 4)


---
## Recap: Where `Guide04` Left Off

Rebuild the training data. `Guide04` found that polynomial terms did not beat the plain baseline, so the feature set here is exactly `Guide03`'s: no polynomial columns.


In [ ]:
from pipeline.ames_workflow import load_ames, clean_ames, split_ames, add_engineered_features, build_preprocessor

X_train, X_test, y_train, y_test = split_ames(clean_ames(load_ames()))
X_train = add_engineered_features(X_train)
print(f"train: {X_train.shape}   test (locked, not used below): {X_test.shape}")


In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline

kf = KFold(n_splits=5, shuffle=True, random_state=42)


def make_model(model, poly_cols=None, degree=2):
    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("prep", build_preprocessor(X_train, poly_cols=poly_cols, degree=degree)),
            ("model", model),
        ]),
        func=np.log, inverse_func=np.exp,
    )


def cv_mae(model, poly_cols=None, degree=2, X=X_train, y=y_train, cv=kf):
    preds = cross_val_predict(make_model(model, poly_cols, degree), X, y, cv=cv)
    return mean_absolute_error(y, preds)


baseline_mae = cv_mae(LinearRegression())
print(f"Guide03/04 baseline (plain LinearRegression, no poly): MAE=${baseline_mae:,.0f}")


---
## Stage 9.1: Cross-Validation, Properly

### One split is not a stable estimate

The baseline above used 5-fold cross-validation. To see why that matters, compare it against five *different single* 75/25 splits of the same training data, same model:


In [ ]:
from sklearn.model_selection import train_test_split

single_split_maes = []
for seed in range(5):
    X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=seed)
    model = make_model(LinearRegression()).fit(X_fit, y_fit)
    mae = mean_absolute_error(y_val, model.predict(X_val))
    single_split_maes.append(mae)
    print(f"  single split, seed={seed}: MAE=${mae:,.0f}")

print(f"\nrange across 5 single splits: ${min(single_split_maes):,.0f} to ${max(single_split_maes):,.0f}")


In [ ]:
kfold_maes = []
for seed in range(5):
    mae = cv_mae(LinearRegression(), cv=KFold(5, shuffle=True, random_state=seed))
    kfold_maes.append(mae)
    print(f"  5-fold CV, fold-seed={seed}: MAE=${mae:,.0f}")

print(f"\nrange across 5 different fold assignments: ${min(kfold_maes):,.0f} to ${max(kfold_maes):,.0f}")


A single 75/25 split's score depends heavily on which quarter of the data happened to land in validation -- a spread of about \$2,200 here. Averaging over 5 folds (still just one way of slicing the data, but five overlapping looks instead of one) tightens that spread to about \$500. Neither number should be treated as exact, but cross-validation's average is the more trustworthy one, and its spread across different fold assignments is itself useful: it is a rough measure of how much to trust any single score that comes out of this notebook.

### Leakage inside cross-validation

Every score above used `cross_val_predict` with a full `Pipeline` -- `build_preprocessor()` gets refit inside every fold, automatically. Two checks on why that discipline matters, even when it looks like unnecessary caution.

**A statistic that is genuinely fold-sensitive.** `Guide02`'s rare-category threshold groups any `Neighborhood` seen fewer than 15 times. Computed on the *full* training set, that is a fixed list. Computed from just one fold's 80%, it can differ:


In [ ]:
full_counts = X_train["Neighborhood"].value_counts()
full_rare = set(full_counts[full_counts < 15].index)
print("rare neighborhoods, using ALL training data:", sorted(full_rare))

for i, (train_idx, _) in enumerate(kf.split(X_train)):
    fold_counts = X_train.iloc[train_idx]["Neighborhood"].value_counts()
    fold_rare = set(fold_counts[fold_counts < 15].index)
    print(f"  fold {i}: {sorted(fold_rare)}  {'(differs)' if fold_rare != full_rare else ''}")


Every fold disagrees with the full-data list -- `Blmngtn`, `SWISU`, and `StoneBr` hover right around the count-15 cutoff, and which side of it they land on depends on exactly which 80% of rows that fold happened to train on. Computing "rare" once on all of training data, then reusing that fixed list inside every fold, would hide this instability rather than fairly accounting for it.

**A statistic that (this time) barely matters.** Compare a scaler fit once on all of `X_train` against one refit inside each fold, holding the model fixed:


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from pipeline.ames_workflow import QUALITY_COLS, QUALITY_SCALE

numeric_cols = list(X_train.select_dtypes("number").columns)
ordinal_cols = [c for c in QUALITY_COLS if c in X_train.columns]
nominal_cols = [c for c in X_train.select_dtypes(exclude="number") if c not in ordinal_cols]

# LEAKY: fit the scaler once, on every training row, before cross-validating anything
leaky_imputer = SimpleImputer(strategy="median").fit(X_train[numeric_cols])
leaky_scaler = StandardScaler().fit(leaky_imputer.transform(X_train[numeric_cols]))
X_train_prescaled = X_train.copy()
X_train_prescaled[numeric_cols] = leaky_scaler.transform(leaky_imputer.transform(X_train[numeric_cols]))

leaky_prep = ColumnTransformer([
    ("num", "passthrough", numeric_cols),  # already scaled outside the loop -- the leak
    ("ord", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("encode", OrdinalEncoder(categories=[QUALITY_SCALE] * len(ordinal_cols)))]), ordinal_cols),
    ("nom", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=15))]), nominal_cols),
])
leaky_model = TransformedTargetRegressor(
    regressor=Pipeline([("prep", leaky_prep), ("model", LinearRegression())]),
    func=np.log, inverse_func=np.exp,
)
leaky_mae = mean_absolute_error(
    y_train, cross_val_predict(leaky_model, X_train_prescaled, y_train, cv=kf)
)
print(f"correct (scaler refit inside every fold): MAE=${baseline_mae:,.0f}")
print(f"leaky   (scaler fit once, outside the CV loop): MAE=${leaky_mae:,.0f}")


Essentially no difference here. With 1,213 training rows, a fold's 80% and the full training set land on almost the same mean and standard deviation, so this particular leak happens to be harmless *this time*. That is worth sitting with for a moment: the two checks above used the same kind of mistake -- fitting something on the full training set instead of inside each fold -- and one exposed a real, visible problem while the other did not. There was no way to know in advance which case this would be. That is exactly why `build_preprocessor()` is always fit inside the `Pipeline`, unconditionally, rather than only when it seems likely to matter.


---
## Stage 9.2: Regularization

$$\text{loss} = \text{prediction error} + \alpha \times \text{penalty}(\beta)$$

Ridge penalizes the sum of squared coefficients; Lasso penalizes the sum of absolute coefficients and can drive some to exactly zero.

### Revisiting `Guide04`'s disaster

`Guide04` expanded every numeric column to degree 2 (921 columns from 37) and watched plain `LinearRegression` produce a numerically meaningless fit. Same feature set, with a Ridge penalty added:


In [ ]:
from sklearn.linear_model import Ridge

all_numeric_cols = list(X_train.select_dtypes("number").columns)

disaster_mae = cv_mae(LinearRegression(), poly_cols=all_numeric_cols, degree=2)
print(f"plain LinearRegression, all 37 cols degree 2:  MAE=${disaster_mae:.3e}  (Guide04's disaster)")

for alpha in (10, 100):
    ridge_mae = cv_mae(Ridge(alpha=alpha), poly_cols=all_numeric_cols, degree=2)
    print(f"Ridge(alpha={alpha:>4}), same 921 columns:     MAE=${ridge_mae:,.0f}")


Ridge does not turn this into the best model in the notebook -- it does not beat the plain baseline either -- but it turns a numerically meaningless result back into an ordinary, usable one. That is what regularization is for: not automatically better accuracy, but a model that behaves sensibly even when the raw feature set is more than the data can support unpenalized.

### Ridge on the actual feature set

Sweep `alpha` on `Guide03`'s real (non-polynomial) features, the ones that go on to `Guide06`:


In [ ]:
ridge_results = {alpha: cv_mae(Ridge(alpha=alpha)) for alpha in (0.1, 1, 3, 10, 30, 100)}
pd.Series(ridge_results, name="CV MAE").rename_axis("alpha")


The minimum is around `alpha=10`, at about \$13,600 -- a genuine improvement over the unregularized baseline's \$14,124, not just a defense against disaster this time.

### Lasso on the same feature set


In [ ]:
import warnings

from sklearn.linear_model import Lasso

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # Lasso's convergence warnings at very small alpha are expected here
    lasso_results = {alpha: cv_mae(Lasso(alpha=alpha, max_iter=30000))
                      for alpha in (0.00003, 0.0001, 0.0003, 0.001, 0.003)}
pd.Series(lasso_results, name="CV MAE").rename_axis("alpha")


Lasso's best (`alpha=0.0003`, about \$13,580) is close enough to Ridge's best to call them a tie. The difference is what Lasso also tells you along the way:


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lasso_fitted = make_model(Lasso(alpha=0.0003, max_iter=30000)).fit(X_train, y_train)

fitted_prep = lasso_fitted.regressor_.named_steps["prep"]
nominal_names = fitted_prep.named_transformers_["nom"].named_steps["encode"].get_feature_names_out(nominal_cols)
feature_names = numeric_cols + QUALITY_COLS + list(nominal_names)
coefficients = pd.Series(lasso_fitted.regressor_.named_steps["model"].coef_, index=feature_names)

survivors = coefficients[coefficients.abs() > 1e-8]
print(f"{len(survivors)} of {len(coefficients)} coefficients survive")
survivors.abs().sort_values(ascending=False).head(8)


`TotalBsmtSF` is one of the columns Lasso zeroes out -- not because basement size does not matter, but because `Guide02`'s engineered `TotalSF` already includes it, and Lasso, unlike Ridge, tends to keep one representative from a correlated group and drop the rest rather than splitting credit between them. That is the collinearity `Guide02` flagged when it introduced `TotalSF`, showing up again here, resolved automatically rather than by hand.


---
## Stage 9.3: Hyperparameter Tuning With Grid Search

The manual sweeps above are exactly what `GridSearchCV` automates -- and it is worth confirming they agree:


In [ ]:
from sklearn.model_selection import GridSearchCV

ridge_grid = GridSearchCV(
    make_model(Ridge()),
    param_grid={"regressor__model__alpha": [1, 3, 10, 30, 100]},
    cv=kf, scoring="neg_mean_absolute_error",
)
ridge_grid.fit(X_train, y_train)
print("Ridge  best:", ridge_grid.best_params_, f" CV MAE=${-ridge_grid.best_score_:,.0f}")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    lasso_grid = GridSearchCV(
        make_model(Lasso(max_iter=30000)),
        param_grid={"regressor__model__alpha": [0.00003, 0.0001, 0.0003, 0.001, 0.003]},
        cv=kf, scoring="neg_mean_absolute_error",
    )
    lasso_grid.fit(X_train, y_train)
print("Lasso  best:", lasso_grid.best_params_, f" CV MAE=${-lasso_grid.best_score_:,.0f}")


Same winners as the manual sweep -- `GridSearchCV` is doing the same cross-validated comparison, just without a hand-written loop. Its convenience is what makes larger grids (many hyperparameters at once) practical, not a different underlying idea.

### Is that score optimistic?

`Guide00` warns that the best score out of several candidates is a little optimistic, because it was picked *for* being the best. A **nested** cross-validation -- an outer loop that re-runs the whole search on each outer fold's training portion -- checks this directly:


In [ ]:
from sklearn.model_selection import cross_val_score

outer_cv = KFold(5, shuffle=True, random_state=1)  # a different split from the search's own cv=kf
nested_scores = cross_val_score(ridge_grid, X_train, y_train, cv=outer_cv, scoring="neg_mean_absolute_error")
print(f"non-nested (search's own best score): MAE=${-ridge_grid.best_score_:,.0f}")
print(f"nested (nested_scores.mean()):        MAE=${-nested_scores.mean():,.0f}")


The nested estimate is a few hundred dollars worse -- a real, if modest, amount of optimism from picking the best of five candidate alphas. That gap is the price of tuning; it is not a sign anything was done wrong.


---
## Freeze the Configuration

**Lasso, `alpha=0.0003`, on `Guide03`'s standard (non-polynomial) feature set.** It ties Ridge on accuracy and additionally identifies 121 of 218 columns as carrying the signal. Its cross-validated score is about \$13,580 MAE; given the nested-CV check above, the honest expectation for genuinely new data is a bit higher than that -- closer to \$13,900-14,000.

This configuration is now frozen. Nothing about it changes based on anything seen after this point.


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    frozen_model = make_model(Lasso(alpha=0.0003, max_iter=30000)).fit(X_train, y_train)
print("frozen_model fitted on all", len(X_train), "training rows; ready for Guide06's final test")


---
## What We Hand to `Guide06`

* **`frozen_model`**: Lasso, `alpha=0.0003`, `Guide03`'s feature set. Not to be refit, retuned, or reconsidered based on the test set.
* **The expected range**: about \$13,600-14,000 MAE, from the cross-validated search and the nested-CV check on it.
* **A caveat for the report**: cross-validation's own "best" score is mildly optimistic; say so alongside the number.
* The locked test set -- opened, for the first and only time, in `Guide06`.


---
## Your Turn

**`encoded_car_data.csv`.** Run a Lasso `alpha` grid search (cross-validated, training data only) on its 35 already-encoded columns. How many of the 35 survive at the best `alpha`? Compare against how few rows (205) this dataset has relative to Ames' 1,213 -- does regularization matter more or less here than it did for Ames, and why?


In [ ]:
# Your turn: load encoded_car_data.csv and grid-search Lasso's alpha.
